# STAT 8017B Project 4 — Data Preprocessing
## Financial Analysis Chatbot (Group 4.1)

This notebook preprocesses all 8 downloaded datasets into clean, model-ready files saved to `data/processed/`.

**Techniques used (from course):**
- NLTK: tokenization, stopword removal, Porter stemming
- scikit-learn: TF-IDF vectorization, StandardScaler, LabelEncoder
- pandas/numpy: missing value handling, type conversion, feature engineering

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

warnings.filterwarnings('ignore')
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

DATA_DIR = os.path.join('..', 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Setup complete. Output directory:', os.path.abspath(PROCESSED_DIR))

Setup complete. Output directory: c:\Users\holli\OneDrive - connect.hku.hk\year6sem2\stat8017\8017project\data\processed


In [2]:
# Shared text preprocessing function (Course Ch.2 — NLTK pipeline)
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Tokenize, lowercase, remove stopwords, apply Porter stemming."""
    tokens = nltk.word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

print('Text cleaning function ready.')
print('Example:', clean_text('The stock market showed strong gains today in technology sector.'))

Text cleaning function ready.
Example: stock market show strong gain today technolog sector


---
## 1. ETFs + Mutual Funds
**Source:** `alpha-insights/ETFs.csv`, `MutualFunds.csv`

Select key columns, handle missing values, scale numerical features.

In [3]:
# --- 1a. ETFs ---
etf_raw = pd.read_csv(os.path.join(DATA_DIR, 'alpha-insights', 'ETFs.csv'))
print(f'ETFs raw: {etf_raw.shape[0]} rows, {etf_raw.shape[1]} columns')

etf_cols = [
    'fund_symbol', 'fund_short_name', 'fund_long_name', 'fund_category', 'fund_family',
    'total_net_assets', 'fund_yield',
    'fund_annual_report_net_expense_ratio',
    'fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
    'fund_return_5years', 'fund_return_10years',
    'fund_mean_annual_return_3years', 'fund_stdev_3years',
    'fund_sharpe_ratio_3years', 'fund_beta_3years',
    'asset_stocks', 'asset_bonds',
    'investment_strategy', 'investment_type', 'size_type'
]
etf = etf_raw[[c for c in etf_cols if c in etf_raw.columns]].copy()

return_cols = ['fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
               'fund_return_5years', 'fund_return_10years']
etf = etf.dropna(subset=return_cols, how='all')

numeric_cols = etf.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    etf[col] = etf[col].fillna(etf[col].median())

etf['investment_strategy'] = etf['investment_strategy'].fillna('')
etf['fund_category'] = etf['fund_category'].fillna('Unknown')
etf['fund_family'] = etf['fund_family'].fillna('Unknown')

# StandardScaler on numerical features (Course Ch.5)
scaler_etf = StandardScaler()
etf_scaled_vals = scaler_etf.fit_transform(etf[numeric_cols])
etf_scaled = etf.copy()
etf_scaled[numeric_cols] = etf_scaled_vals

etf.to_csv(os.path.join(PROCESSED_DIR, 'etf_clean.csv'), index=False)
joblib.dump(scaler_etf, os.path.join(PROCESSED_DIR, 'etf_scaler.joblib'))
print(f'ETFs cleaned: {etf.shape[0]} rows, {etf.shape[1]} columns')
print(f'Numeric columns scaled: {numeric_cols}')
etf.head()

ETFs raw: 2310 rows, 142 columns
ETFs cleaned: 2116 rows, 22 columns
Numeric columns scaled: ['total_net_assets', 'fund_yield', 'fund_annual_report_net_expense_ratio', 'fund_return_ytd', 'fund_return_1year', 'fund_return_3years', 'fund_return_5years', 'fund_return_10years', 'fund_mean_annual_return_3years', 'fund_stdev_3years', 'fund_sharpe_ratio_3years', 'fund_beta_3years', 'asset_stocks', 'asset_bonds']


,fund_symbol,fund_short_name,fund_long_name,fund_category,fund_family,total_net_assets,fund_yield,fund_annual_report_net_expense_ratio,fund_return_ytd,fund_return_1year,...,fund_return_10years,fund_mean_annual_return_3years,fund_stdev_3years,fund_sharpe_ratio_3years,fund_beta_3years,asset_stocks,asset_bonds,investment_strategy,investment_type,size_type
0,AAAU,DWS RREEF Real Assets Fund - Cl,DWS RREEF Real Assets Fund - Class A,Unknown,DWS,3.844486e+08,0.0157,0.0018,-0.0465,-0.0790,...,0.0542,1.23,14.93,0.91,0.07,0.9979,0.0,The investment seeks total return in excess of...,NaN,NaN
1,AADR,AllianzGI Health Sciences Fund,Virtus AllianzGI Health Sciences Fund Class P,Foreign Large Growth,Virtus,8.883616e+07,0.0031,0.0110,0.0940,0.2587,...,0.0830,0.85,22.42,0.40,1.11,0.9979,0.0,The investment seeks long-term capital appreci...,Blend,Large
2,AAXJ,NaN,American Century One Choice Blend+ 2015 Portfo...,Pacific/Asia ex-Japan Stk,American Century Investments,5.574672e+09,0.0110,0.0070,-0.0173,0.1859,...,0.0535,0.80,18.48,0.46,0.90,0.9979,0.0,The investment seeks the highest total return ...,Blend,Large
3,ABEQ,Thrivent Large Cap Growth Fund,Thrivent Large Cap Growth Fund Class A,Large Value,Thrivent Funds,4.969417e+07,0.0049,0.0088,0.0700,0.2252,...,0.0542,0.88,19.66,0.56,1.02,0.9979,0.0,The investment seeks long-term capital appreci...,Value,Large
4,ACES,NaN,American Century One Choice Blend+ 2015 Portfo...,Miscellaneous Sector,American Century Investments,1.007483e+09,0.0053,0.0055,-0.0511,0.9507,...,0.0542,3.65,32.60,1.30,1.31,0.9992,0.0,The investment seeks the highest total return ...,Growth,Medium


In [4]:
# --- 1b. Mutual Funds ---
mf_raw = pd.read_csv(os.path.join(DATA_DIR, 'alpha-insights', 'MutualFunds.csv'))
print(f'Mutual Funds raw: {mf_raw.shape[0]} rows, {mf_raw.shape[1]} columns')

mf_cols = [
    'fund_symbol', 'fund_short_name', 'fund_long_name', 'fund_category', 'fund_family',
    'total_net_assets', 'fund_yield',
    'fund_annual_report_net_expense_ratio',
    'fund_return_ytd', 'fund_return_1year', 'fund_return_3years',
    'fund_return_5years', 'fund_return_10years',
    'fund_mean_annual_return_3years', 'fund_stdev_3years',
    'fund_sharpe_ratio_3years', 'fund_beta_3years',
    'investment_strategy', 'investment_type', 'size_type'
]
mf = mf_raw[[c for c in mf_cols if c in mf_raw.columns]].copy()

mf = mf.dropna(subset=return_cols, how='all')

mf_numeric = mf.select_dtypes(include=[np.number]).columns.tolist()
for col in mf_numeric:
    mf[col] = mf[col].fillna(mf[col].median())

mf['investment_strategy'] = mf['investment_strategy'].fillna('')
mf['fund_category'] = mf['fund_category'].fillna('Unknown')
mf['fund_family'] = mf['fund_family'].fillna('Unknown')

mf.to_csv(os.path.join(PROCESSED_DIR, 'mutualfund_clean.csv'), index=False)
print(f'Mutual Funds cleaned: {mf.shape[0]} rows, {mf.shape[1]} columns')
mf.head()

Mutual Funds raw: 23783 rows, 298 columns
Mutual Funds cleaned: 23382 rows, 20 columns


,fund_symbol,fund_short_name,fund_long_name,fund_category,fund_family,total_net_assets,fund_yield,fund_annual_report_net_expense_ratio,fund_return_ytd,fund_return_1year,fund_return_3years,fund_return_5years,fund_return_10years,fund_mean_annual_return_3years,fund_stdev_3years,fund_sharpe_ratio_3years,fund_beta_3years,investment_strategy,investment_type,size_type
0,AAAAX,DWS RREEF Real Assets Fund - Cl,DWS RREEF Real Assets Fund - Class A,World Allocation,DWS,2.979347e+09,0.0186,0.0122,0.21026,0.22970,0.13811,0.09078,0.06058,0.91,13.68,0.71,1.12,The investment seeks total return in excess of...,Value,Large
1,AAAEX,AllianzGI Health Sciences Fund,Virtus AllianzGI Health Sciences Fund Class P,Health,Virtus,1.953489e+08,0.0135,0.0109,0.19077,0.23934,0.09110,0.07830,0.07850,1.49,15.82,1.05,0.71,The investment seeks long-term capital appreci...,Blend,Large
3,AAAGX,Thrivent Large Cap Growth Fund,Thrivent Large Cap Growth Fund Class A,Large Growth,Thrivent Funds,2.078607e+09,0.0135,0.0108,0.24559,0.30705,0.31791,0.19264,0.19622,2.03,20.21,1.14,1.03,The investment seeks long-term capital appreci...,Growth,Large
5,AAAIX,American Century Strategic Allo,American Century Strategic Allocation: Aggress...,Allocation--70% to 85% Equity,American Century Investments,8.483190e+08,0.0083,0.0063,0.15050,0.19515,0.17981,0.11902,0.11973,1.22,15.74,0.85,1.37,The investment seeks the highest level of tota...,Blend,Large
10,AAANX,Horizon Active Asset Allocation,Horizon Active Asset Allocation Fund Investor ...,Tactical Allocation,Horizon Investments,7.045766e+08,0.0041,0.0151,0.20491,0.24963,0.16541,0.11116,0.07850,1.04,19.32,0.58,1.67,The investment seeks capital appreciation. The...,Blend,Large


---
## 2. Customer Complaints
**Source:** `complaints/customer_complaints.csv`

Sample 200K rows (stratified by Product), NLTK text preprocessing, TF-IDF vectorization, label encode Product.

In [5]:
complaints_raw = pd.read_csv(os.path.join(DATA_DIR, 'complaints', 'customer_complaints.csv'))
print(f'Complaints raw: {complaints_raw.shape[0]:,} rows')
print(f'Products: {complaints_raw["Product"].nunique()} categories')
print(complaints_raw['Product'].value_counts().head(10))

Complaints raw: 1,473,407 rows
Products: 18 categories
Product
Credit reporting, credit repair services, or other personal consumer reports    323549
Mortgage                                                                        293396
Debt collection                                                                 275759
Credit reporting                                                                140429
Credit card                                                                      89190
Bank account or service                                                          86205
Credit card or prepaid card                                                      65472
Student loan                                                                     55854
Checking or savings account                                                      55709
Consumer Loan                                                                    31575
Name: count, dtype: int64


In [6]:
SAMPLE_SIZE = 200_000

comp = complaints_raw.dropna(subset=['Product', 'Issue']).copy()

# Stratified sample to preserve product category proportions
if len(comp) > SAMPLE_SIZE:
    comp, _ = train_test_split(
        comp, train_size=SAMPLE_SIZE, random_state=42, stratify=comp['Product']
    )
print(f'Sampled {len(comp):,} rows (stratified by Product)')

comp['text'] = comp['Issue'].fillna('') + ' ' + comp['Sub-issue'].fillna('')
print(f'Applying NLTK text cleaning to {len(comp):,} rows (this will take a few minutes)...')
comp['cleaned_text'] = comp['text'].apply(clean_text)

le_product = LabelEncoder()
comp['product_label'] = le_product.fit_transform(comp['Product'])

tfidf_complaints = TfidfVectorizer(max_features=5000)
X_complaints = tfidf_complaints.fit_transform(comp['cleaned_text'])

comp_save = comp[['Complaint ID', 'Product', 'product_label', 'Issue', 'Sub-issue',
                   'text', 'cleaned_text', 'Company', 'State', 'Date received']].copy()
comp_save.to_csv(os.path.join(PROCESSED_DIR, 'complaints_clean.csv'), index=False)
joblib.dump(tfidf_complaints, os.path.join(PROCESSED_DIR, 'complaints_tfidf.joblib'))
joblib.dump(le_product, os.path.join(PROCESSED_DIR, 'complaints_labels.joblib'))
joblib.dump(X_complaints, os.path.join(PROCESSED_DIR, 'complaints_tfidf_matrix.joblib'))

print(f'Complaints cleaned: {comp_save.shape[0]:,} rows')
print(f'TF-IDF matrix shape: {X_complaints.shape}')
print(f'Product labels: {list(le_product.classes_)}')
comp_save.head()

Sampled 200,000 rows (stratified by Product)
Applying NLTK text cleaning to 200,000 rows (this will take a few minutes)...
Complaints cleaned: 200,000 rows
TF-IDF matrix shape: (200000, 332)
Product labels: ['Bank account or service', 'Checking or savings account', 'Consumer Loan', 'Credit card', 'Credit card or prepaid card', 'Credit reporting', 'Credit reporting, credit repair services, or other personal consumer reports', 'Debt collection', 'Money transfer, virtual currency, or money service', 'Money transfers', 'Mortgage', 'Other financial service', 'Payday loan', 'Payday loan, title loan, or personal loan', 'Prepaid card', 'Student loan', 'Vehicle loan or lease', 'Virtual currency']


,Complaint ID,Product,product_label,Issue,Sub-issue,text,cleaned_text,Company,State,Date received
1044383,1211488,Mortgage,10,"Loan servicing, payments, escrow account",NaN,"Loan servicing, payments, escrow account",loan servic payment escrow account,"CITIZENS FINANCIAL GROUP, INC.",WA,2015-01-27
926748,2279442,Credit reporting,5,Incorrect information on credit report,Information is not mine,Incorrect information on credit report Informa...,incorrect inform credit report inform mine,Experian Information Solutions Inc.,TX,2017-01-07
1100200,1605378,Credit reporting,5,Incorrect information on credit report,Account status,Incorrect information on credit report Account...,incorrect inform credit report account statu,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CT,2015-10-14
1455869,1288234,Debt collection,7,Cont'd attempts collect debt not owed,Debt is not mine,Cont'd attempts collect debt not owed Debt is ...,cont attempt collect debt owe debt mine,"Diversified Consultants, Inc.",CA,2015-03-18
79726,3143669,"Credit reporting, credit repair services, or o...",6,Improper use of your report,Credit inquiries on your report that you don't...,Improper use of your report Credit inquiries o...,improp use report credit inquiri report recogn,Experian Information Solutions Inc.,FL,2019-02-05


---
## 3. Financial Q&A Knowledge Base
**Source:** `financial-qa/Financial-QA-10k.csv`

Clean Q&A text, build TF-IDF matrix on questions for cosine-similarity retrieval.

In [7]:
qa_raw = pd.read_csv(os.path.join(DATA_DIR, 'financial-qa', 'Financial-QA-10k.csv'))
print(f'Financial Q&A raw: {qa_raw.shape[0]:,} rows')
print(f'Columns: {list(qa_raw.columns)}')

qa = qa_raw.dropna(subset=['question', 'answer']).copy()
qa['cleaned_question'] = qa['question'].apply(clean_text)
qa['cleaned_answer'] = qa['answer'].apply(clean_text)

# Build TF-IDF on questions for retrieval (Course Ch.2 — cosine similarity)
tfidf_qa = TfidfVectorizer(max_features=5000)
qa_tfidf_matrix = tfidf_qa.fit_transform(qa['cleaned_question'])

# Save
qa.to_csv(os.path.join(PROCESSED_DIR, 'qa_clean.csv'), index=False)
joblib.dump(tfidf_qa, os.path.join(PROCESSED_DIR, 'qa_tfidf_vectorizer.joblib'))
joblib.dump(qa_tfidf_matrix, os.path.join(PROCESSED_DIR, 'qa_tfidf_matrix.joblib'))

print(f'Q&A cleaned: {qa.shape[0]:,} rows')
print(f'TF-IDF matrix shape: {qa_tfidf_matrix.shape}')
print(f'Tickers covered: {qa["ticker"].nunique()}')
qa[['question', 'answer', 'ticker']].head()

Financial Q&A raw: 7,000 rows
Columns: ['question', 'answer', 'context', 'ticker', 'filing']
Q&A cleaned: 6,997 rows
TF-IDF matrix shape: (6997, 3114)
Tickers covered: 69


,question,answer,ticker
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,NVDA
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,NVDA
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,NVDA
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,NVDA
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,NVDA


In [8]:
# Quick retrieval test: cosine similarity (Course Ch.2)
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_answer(query, top_k=3):
    """Find the top-k most similar Q&A pairs for a given query."""
    query_vec = tfidf_qa.transform([clean_text(query)])
    similarities = cosine_similarity(query_vec, qa_tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]
    results = []
    for idx in top_indices:
        results.append({
            'score': round(similarities[idx], 4),
            'question': qa.iloc[idx]['question'],
            'answer': qa.iloc[idx]['answer'][:200] + '...'
        })
    return results

test_results = retrieve_answer('What is the company revenue?')
for r in test_results:
    print(f"Score: {r['score']}")
    print(f"Q: {r['question'][:100]}")
    print(f"A: {r['answer'][:150]}")
    print('---')

Score: 0.6758
Q: What were the total net revenues of the company in 2023?
A: $32,653 million...
---
Score: 0.6006
Q: What type of revenue does 'Non-advertising revenue' include?
A: Non-advertising revenue includes RL revenue from consumer hardware products and FoA Other revenue, including revenue from WhatsApp Business Platform, 
---
Score: 0.583
Q: How did FedEx's revenue in 2023 compare to its revenue in 2022?
A: FedEx's revenue decreased by 4%, going from $93,512 million in 2022 to $90,155 million in 2023....
---


---
## 4. Financial Phrasebank (Sentiment)
**Source:** `phrasebank/all-data.csv`

Parse sentiment labels, NLTK text preprocessing, train/test split.

In [9]:
pb_raw = pd.read_csv(
    os.path.join(DATA_DIR, 'phrasebank', 'all-data.csv'),
    encoding='latin-1', header=None, names=['sentiment', 'sentence']
)
print(f'Phrasebank raw: {pb_raw.shape[0]:,} rows')
print(pb_raw['sentiment'].value_counts())

pb = pb_raw.dropna().copy()
pb['cleaned_text'] = pb['sentence'].apply(clean_text)

# Encode sentiment labels
le_sentiment = LabelEncoder()
pb['sentiment_label'] = le_sentiment.fit_transform(pb['sentiment'])

# Train/test split (80/20 stratified) — Course Ch.3
pb_train, pb_test = train_test_split(pb, test_size=0.2, random_state=42,
                                      stratify=pb['sentiment_label'])

pb.to_csv(os.path.join(PROCESSED_DIR, 'phrasebank_clean.csv'), index=False)
joblib.dump(le_sentiment, os.path.join(PROCESSED_DIR, 'phrasebank_label_encoder.joblib'))

print(f'Phrasebank cleaned: {pb.shape[0]:,} rows')
print(f'Train: {len(pb_train):,}, Test: {len(pb_test):,}')
print(f'Labels: {list(le_sentiment.classes_)}')
pb.head()

Phrasebank raw: 4,846 rows
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
Phrasebank cleaned: 4,846 rows
Train: 3,876, Test: 970
Labels: ['negative', 'neutral', 'positive']


,sentiment,sentence,cleaned_text,sentiment_label
0,neutral,"According to Gran , the company has no plans t...",accord gran compani plan move product russia a...,1
1,neutral,Technopolis plans to develop in stages an area...,technopoli plan develop stage area less squar ...,1
2,negative,The international electronic industry company ...,intern electron industri compani elcoteq laid ...,0
3,positive,With the new production plant the company woul...,new product plant compani would increas capac ...,2
4,positive,According to the company 's updated strategy f...,accord compani updat strategi year baswar targ...,2


---
## 5. FinSen Financial Sentiment
**Source:** `finsen/FinSen_US_Categorized.csv`

NLTK preprocessing on Content, encode Category for topic classification.

In [10]:
finsen_raw = pd.read_csv(os.path.join(DATA_DIR, 'finsen', 'FinSen_US_Categorized.csv'))
print(f'FinSen raw: {finsen_raw.shape[0]:,} rows')
print(f'Categories: {finsen_raw["Category"].nunique()}')
print(finsen_raw['Category'].value_counts())

finsen = finsen_raw.dropna(subset=['Content', 'Category']).copy()
print(f'\nApplying NLTK text cleaning to {len(finsen):,} rows...')
finsen['cleaned_content'] = finsen['Content'].apply(clean_text)

# Label encode Category for classification
le_finsen = LabelEncoder()
finsen['category_label'] = le_finsen.fit_transform(finsen['Category'])

# TF-IDF on content
tfidf_finsen = TfidfVectorizer(max_features=5000)
X_finsen = tfidf_finsen.fit_transform(finsen['cleaned_content'])

finsen.to_csv(os.path.join(PROCESSED_DIR, 'finsen_clean.csv'), index=False)
joblib.dump(tfidf_finsen, os.path.join(PROCESSED_DIR, 'finsen_tfidf.joblib'))
joblib.dump(le_finsen, os.path.join(PROCESSED_DIR, 'finsen_label_encoder.joblib'))
joblib.dump(X_finsen, os.path.join(PROCESSED_DIR, 'finsen_tfidf_matrix.joblib'))

print(f'FinSen cleaned: {finsen.shape[0]:,} rows')
print(f'TF-IDF matrix: {X_finsen.shape}')
print(f'Category labels: {list(le_finsen.classes_)}')
finsen.head()

FinSen raw: 15,534 rows
Categories: 12
Category
Market Indicators                5708
Corporate Finance                2775
Economic Health                  1213
Trade and Commerce               1171
Credit and Lending               1171
Production and Utilization        905
Housing Market                    877
Consumer and Services             638
Government and Policy             447
Inflation and Prices              421
Labor Market Conditions Index     168
Other                              40
Name: count, dtype: int64

Applying NLTK text cleaning to 15,534 rows...
FinSen cleaned: 15,534 rows
TF-IDF matrix: (15534, 5000)
Category labels: ['Consumer and Services', 'Corporate Finance', 'Credit and Lending', 'Economic Health', 'Government and Policy', 'Housing Market', 'Inflation and Prices', 'Labor Market Conditions Index', 'Market Indicators', 'Other', 'Production and Utilization', 'Trade and Commerce']


,Title,Tag,Content,Category,cleaned_content,category_label
0,"TSX Slightly Down, Books Weekly Gains",Stock Market,"TSX Slightly Down, Books Weekly GainsUnited St...",Market Indicators,tsx slightli book weekli gainsunit state stock...,8
1,UnitedHealth Hits 4-week High,stocks,UnitedHealth Hits 4-week HighUnited States sto...,Market Indicators,unitedhealth hit highunit state stocksunitedhe...,8
2,Cisco Systems Hits 4-week Low,stocks,Cisco Systems Hits 4-week LowUnited States sto...,Market Indicators,cisco system hit lowunit state stockscisco sys...,8
3,AT&T Hits All-time Low,stocks,AT&T Hits All-time LowUnited States stocksAT&T...,Market Indicators,hit lowunit state stocksat decreas low day ago,8
4,Microsoft Hits 4-week High,stocks,Microsoft Hits 4-week HighUnited States stocks...,Market Indicators,microsoft hit highunit state stocksmicrosoft i...,8


---
## 6. Financial News
**Source:** `financial-news/financial_news_events.csv`

Drop empty headlines, parse dates, NLTK preprocessing.

In [11]:
news_raw = pd.read_csv(os.path.join(DATA_DIR, 'financial-news', 'financial_news_events.csv'))
print(f'Financial News raw: {news_raw.shape[0]:,} rows')
print(f'Columns: {list(news_raw.columns)}')

news = news_raw.dropna(subset=['Headline']).copy()
news['Date'] = pd.to_datetime(news['Date'], errors='coerce')
news['cleaned_headline'] = news['Headline'].apply(clean_text)

# Fill missing Sentiment as 'Unknown' (can predict later with trained model)
news['Sentiment'] = news['Sentiment'].fillna('Unknown')
news['Sector'] = news['Sector'].fillna('Unknown')
news['Impact_Level'] = news['Impact_Level'].fillna('Unknown')

# Convert Index_Change_Percent to numeric
news['Index_Change_Percent'] = pd.to_numeric(news['Index_Change_Percent'], errors='coerce')

news.to_csv(os.path.join(PROCESSED_DIR, 'news_clean.csv'), index=False)
print(f'News cleaned: {news.shape[0]:,} rows')
print(f'Sentiment distribution:\n{news["Sentiment"].value_counts()}')
news.head()

Financial News raw: 3,024 rows
Columns: ['Date', 'Headline', 'Source', 'Market_Event', 'Market_Index', 'Index_Change_Percent', 'Trading_Volume', 'Sentiment', 'Sector', 'Impact_Level', 'Related_Company', 'News_Url']
News cleaned: 2,876 rows
Sentiment distribution:
Sentiment
Negative    926
Neutral     906
Positive    882
Unknown     162
Name: count, dtype: int64


,Date,Headline,Source,Market_Event,Market_Index,Index_Change_Percent,Trading_Volume,Sentiment,Sector,Impact_Level,Related_Company,News_Url,cleaned_headline
0,2025-05-21,Nikkei 225 index benefits from a weaker yen,Times of India,Commodity Price Shock,DAX,3.52,166.45,Unknown,Technology,High,Goldman Sachs,https://timesofindia.indiatimes.com/business/m...,nikkei index benefit weaker yen
1,2025-05-18,Government subsidy program gives a lift to the...,Financial Times,Central Bank Meeting,Shanghai Composite,-3.39,57.61,Unknown,Retail,Low,ExxonMobil,https://timesofindia.indiatimes.com/business/m...,govern subsidi program give lift agricultur se...
2,2025-06-25,New housing data release shows a slowdown in m...,The Hindu Business Line,Consumer Confidence Report,Shanghai Composite,-0.05,403.22,Neutral,Retail,Medium,Boeing,https://www.moneycontrol.com/us-markets/sp-500,new hous data releas show slowdown market activ
3,2025-07-21,Massive stock buyback program announced by a c...,The Economist,Commodity Price Shock,NSE Nifty,-2.29,100.11,Positive,Consumer Goods,Low,Samsung Electronics,https://www.cnbc.com/2025/09/automotive-indust...,massiv stock buyback program announc consum go...
4,2025-07-23,Government spending bill is expected to stimul...,The Motley Fool,Inflation Data Release,Nasdaq Composite,-3.97,438.22,Negative,Consumer Goods,Low,JP Morgan Chase,https://www.bloomberg.com/australia/asx-200-pe...,govern spend bill expect stimul economi


---
## 7. Finance Survey Data
**Source:** `finance-data/Finance_data.csv`

Light cleaning, encode categorical columns.

In [12]:
survey_raw = pd.read_csv(os.path.join(DATA_DIR, 'finance-data', 'Finance_data.csv'))
print(f'Survey raw: {survey_raw.shape[0]} rows, {survey_raw.shape[1]} columns')

survey = survey_raw.copy()

# Encode categorical columns
cat_cols = survey.select_dtypes(include=['object']).columns.tolist()
label_encoders_survey = {}
for col in cat_cols:
    le = LabelEncoder()
    survey[col + '_encoded'] = le.fit_transform(survey[col].fillna('Unknown'))
    label_encoders_survey[col] = le

survey.to_csv(os.path.join(PROCESSED_DIR, 'survey_clean.csv'), index=False)
joblib.dump(label_encoders_survey, os.path.join(PROCESSED_DIR, 'survey_encoders.joblib'))
print(f'Survey cleaned: {survey.shape[0]} rows, {survey.shape[1]} columns')
print(f'Encoded columns: {cat_cols}')
survey.head()

Survey raw: 40 rows, 24 columns
Survey cleaned: 40 rows, 40 columns
Encoded columns: ['gender', 'Investment_Avenues', 'Stock_Marktet', 'Factor', 'Objective', 'Purpose', 'Duration', 'Invest_Monitor', 'Expect', 'Avenue', 'What are your savings objectives?', 'Reason_Equity', 'Reason_Mutual', 'Reason_Bonds', 'Reason_FD', 'Source']


,gender,age,Investment_Avenues,Mutual_Funds,Equity_Market,Debentures,Government_Bonds,Fixed_Deposits,PPF,Gold,...,Duration_encoded,Invest_Monitor_encoded,Expect_encoded,Avenue_encoded,What are your savings objectives?_encoded,Reason_Equity_encoded,Reason_Mutual_encoded,Reason_Bonds_encoded,Reason_FD_encoded,Source_encoded
0,Female,34,Yes,1,2,5,3,7,6,4,...,0,1,1,2,2,0,0,1,0,2
1,Female,23,Yes,4,3,2,1,5,6,7,...,3,2,1,2,1,1,0,1,1,0
2,Male,30,Yes,3,6,4,2,5,1,7,...,1,0,1,0,2,0,2,0,0,3
3,Male,22,Yes,2,1,3,7,6,4,5,...,2,0,0,0,2,1,1,2,1,1
4,Female,24,No,2,1,3,6,4,5,7,...,2,0,1,0,2,0,0,1,2,1


---
## 8. S&P 500 / ETF / Crypto Prices
**Source:** `sp500-etf-crypto/` (~2,767 individual CSV files)

Scan ALL CSV files in the directory, skip empty ones, merge, compute daily returns and moving averages.

In [18]:
PRICE_DIR = os.path.join(DATA_DIR, 'sp500-etf-crypto')
MIN_FILE_BYTES = 100  # skip header-only / empty CSVs (42 bytes)

csv_files = sorted([f for f in os.listdir(PRICE_DIR) if f.endswith('.csv')])
print(f'Found {len(csv_files)} CSV files in {PRICE_DIR}')

frames = []
skipped = 0
for fname in csv_files:
    fpath = os.path.join(PRICE_DIR, fname)
    if os.path.getsize(fpath) < MIN_FILE_BYTES:
        skipped += 1
        continue
    try:
        df_t = pd.read_csv(fpath)
        if len(df_t) == 0:
            skipped += 1
            continue
        df_t['ticker'] = fname.replace('.csv', '')
        frames.append(df_t)
    except Exception:
        skipped += 1

prices = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(frames):,} tickers, skipped {skipped} empty/bad files')
print(f'Total rows: {prices.shape[0]:,}')

Found 2766 CSV files in ..\data\sp500-etf-crypto
Loaded 2,231 tickers, skipped 535 empty/bad files
Total rows: 6,246,227


In [19]:
prices['Date'] = pd.to_datetime(prices['Date'], errors='coerce')
prices = prices.sort_values(['ticker', 'Date']).reset_index(drop=True)

price_col = 'Close' if 'Close' in prices.columns else 'Adj Close'

prices['daily_return'] = prices.groupby('ticker')[price_col].pct_change()

prices['ma_20'] = prices.groupby('ticker')[price_col].transform(
    lambda x: x.rolling(window=20, min_periods=1).mean()
)
prices['ma_50'] = prices.groupby('ticker')[price_col].transform(
    lambda x: x.rolling(window=50, min_periods=1).mean()
)

fill_cols = [c for c in prices.columns if c not in ['ticker', 'Date']]
prices[fill_cols] = prices.groupby('ticker')[fill_cols].ffill()

prices.to_csv(os.path.join(PROCESSED_DIR, 'prices_clean.csv'), index=False)
print(f'Prices cleaned: {prices.shape[0]:,} rows, {prices.shape[1]} columns')
print(f'Tickers: {prices["ticker"].nunique():,}')
print(f'Date range: {prices["Date"].min()} to {prices["Date"].max()}')
prices.head(10)

Prices cleaned: 6,246,227 rows, 11 columns
Tickers: 2,231
Date range: 1962-01-02 00:00:00 to 2025-02-05 00:00:00


,Date,Adj Close,Close,High,Low,Open,Volume,ticker,daily_return,ma_20,ma_50
0,1999-11-18,26.511736,31.473534,35.765381,28.612303,32.546494,62546380.0,A,NaN,31.473534,31.473534
1,1999-11-19,24.327528,28.880545,30.758226,28.478184,30.713518,15234146.0,A,-0.082386,30.177039,30.177039
2,1999-11-22,26.511736,31.473534,31.473534,28.657009,29.551144,6577870.0,A,0.089783,30.609204,30.609204
3,1999-11-23,24.101582,28.612303,31.205294,28.612303,30.400572,5975611.0,A,-0.090909,30.109979,30.109979
4,1999-11-24,24.741781,29.372318,29.998213,28.612303,28.701717,4843231.0,A,0.026563,29.962447,29.962447
5,1999-11-26,24.817099,29.461731,29.685265,29.148785,29.238197,1729466.0,A,0.003044,29.878994,29.878994
6,1999-11-29,25.381966,30.132332,30.355865,29.014664,29.327612,4074751.0,A,0.022762,29.915185,29.915185
7,1999-11-30,25.419634,30.177038,30.713518,29.282904,30.042917,4310034.0,A,0.001484,29.947917,29.947917
8,1999-12-01,25.871540,30.713518,31.071173,29.953505,30.177038,2957329.0,A,0.017778,30.032984,30.032984
9,1999-12-02,26.587061,31.562946,32.188843,30.892345,31.294706,3069868.0,A,0.027656,30.185980,30.185980


---
## Summary of Preprocessed Files
List all files saved to `data/processed/` with their sizes.

In [20]:
print('=' * 60)
print('PREPROCESSING COMPLETE — Output files:')
print('=' * 60)
total_size = 0
for f in sorted(os.listdir(PROCESSED_DIR)):
    fpath = os.path.join(PROCESSED_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    total_size += size_mb
    print(f'  {f:45s} {size_mb:8.1f} MB')
print('-' * 60)
print(f'  {"TOTAL":45s} {total_size:8.1f} MB')
print('=' * 60)

PREPROCESSING COMPLETE — Output files:
  complaints_clean.csv                              48.1 MB
  complaints_labels.joblib                           0.0 MB
  complaints_tfidf.joblib                            0.0 MB
  complaints_tfidf_matrix.joblib                    13.0 MB
  etf_clean.csv                                      1.7 MB
  etf_scaler.joblib                                  0.0 MB
  finsen_clean.csv                                  13.8 MB
  finsen_label_encoder.joblib                        0.0 MB
  finsen_tfidf.joblib                                0.2 MB
  finsen_tfidf_matrix.joblib                         6.9 MB
  mutualfund_clean.csv                              18.1 MB
  news_clean.csv                                     0.8 MB
  phrasebank_clean.csv                               1.1 MB
  phrasebank_label_encoder.joblib                    0.0 MB
  prices_clean.csv                                1009.2 MB
  qa_clean.csv                                       4.1 MB
 